# LFW Grad-CAM — 04. Step 2 compression characterization

동일한 `origin_embedding_artifact_uid`의 512D만 PCA-only 또는 PQ-only
정량 runner에 전달합니다. 이 노트북은 PCA/PQ fitting 로직을 복제하지 않고,
runner 결과가 전체 표본·모델 provenance와 fallback-free 조건을 지키는지
검사해 lineage 열을 붙입니다.

현재 전용 PyTorch Step 2 정량 runner가 아직 없으므로 실제 runner 출력
경로가 없으면 의도적으로 중단합니다.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(C:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

MODEL_PROFILE = "arcface_ms1mv3_r100"     # arcface, adaface, magface 중 이번 실행 profile
MODE = "dev"               # 빠른 검증은 dev, 전체 논문 실행만 real
DATA_FRACTION = 0.10       # identity 단위 사용 비율; 0 < 값 <= 1
SEED = 42                  # 부분집합과 random control의 재현 seed
EXECUTE_STAGE = False      # 필수 입력을 채우고 이 단계 계산 시에만 True
WRITE_OUTPUTS = False      # 새 immutable artifact 저장 시에만 True

all_profiles = CONFIG["models"]["selected_profiles"] + CONFIG["models"].get("bridge_profiles", [])
available_profiles = CONFIG["models"]["profiles"]
blocked_profiles = CONFIG["models"].get("blocked_profiles", [])

if MODEL_PROFILE in blocked_profiles:
    raise RuntimeError(f"차단된 profile입니다: {MODEL_PROFILE}")
if MODEL_PROFILE not in available_profiles:
    raise ValueError(f"지원하지 않는 MODEL_PROFILE: {MODEL_PROFILE}")
PROFILE = available_profiles[MODEL_PROFILE]
MODEL_FAMILY = PROFILE["family"]
if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if MODE == "real" and DATA_FRACTION != 1.0:
    raise ValueError("real 모드는 DATA_FRACTION=1.0이어야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")


In [ ]:
import pandas as pd

from research.evaluation import annotate_compression_lineage
from research.explainability.gradcam import (
    read_prepared_population_artifact,
)

PREPARED_ARTIFACT_DIR = None
PAIRED_METRICS_PATH = None
RETRIEVAL_METRICS_PATH = None
LINEAGED_PAIRED_OUTPUT_PATH = None
LINEAGED_RETRIEVAL_OUTPUT_PATH = None


In [ ]:
if EXECUTE_STAGE:
    paths = {
        "prepared": PREPARED_ARTIFACT_DIR,
        "paired": PAIRED_METRICS_PATH,
        "retrieval": RETRIEVAL_METRICS_PATH,
    }
    missing = [name for name, value in paths.items() if value is None]
    if missing:
        raise RuntimeError(
            "전용 PyTorch Step 2 정량 runner 출력이 필요합니다: "
            f"{missing}"
        )
    prepared = read_prepared_population_artifact(paths["prepared"])
    paired = pd.read_parquet(paths["paired"])
    retrieval = pd.read_parquet(paths["retrieval"])
    for name, frame in (("paired", paired), ("retrieval", retrieval)):
        if "origin_fallback_used" not in frame:
            raise ValueError(f"{name}에 origin_fallback_used가 없습니다.")
        if frame["origin_fallback_used"].fillna(True).astype(bool).any():
            raise ValueError(f"{name}에 origin fallback 사용 행이 있습니다.")

    allowed_dimensions = {384, 256, 128, 64, 32}
    pca = paired.loc[paired["compression_family"].eq("pca")]
    observed_dimensions = set(
        pd.to_numeric(pca["search_dimension"], errors="raise").astype(int)
    )
    if not observed_dimensions.issubset(allowed_dimensions):
        raise ValueError("정의되지 않은 PCA 차원이 포함됐습니다.")
    if "pca_pq" in set(paired["compression_family"].astype(str)):
        raise ValueError("PCA→PQ 연쇄는 이 연구의 압축군이 아닙니다.")

    lineage = {
        "extraction_uid": prepared.extraction_uid,
        "dataset_id": prepared.dataset_id,
        "model_uid": prepared.model_uid,
        "origin_embedding_artifact_uid": (
            prepared.origin_embedding_artifact_uid
        ),
    }
    paired = annotate_compression_lineage(paired, **lineage)
    retrieval = annotate_compression_lineage(retrieval, **lineage)
    compression_summary = {
        "paired_rows": int(len(paired)),
        "retrieval_rows": int(len(retrieval)),
        "families": sorted(
            paired["compression_family"].astype(str).unique()
        ),
        "profiles": sorted(
            paired["compression_profile"].astype(str).unique()
        ),
        **lineage,
    }
    if WRITE_OUTPUTS:
        outputs = {
            "paired": LINEAGED_PAIRED_OUTPUT_PATH,
            "retrieval": LINEAGED_RETRIEVAL_OUTPUT_PATH,
        }
        if any(value is None for value in outputs.values()):
            raise RuntimeError("두 lineage 출력 경로를 지정하세요.")
        for name, frame in (("paired", paired), ("retrieval", retrieval)):
            destination = Path(outputs[name]).resolve()
            if destination.exists():
                raise FileExistsError(
                    f"기존 결과를 덮어쓸 수 없습니다: {destination}"
                )
            destination.parent.mkdir(parents=True, exist_ok=True)
            frame.to_parquet(destination, index=False)
else:
    compression_summary = {
        "status": "not_executed",
        "reason": "EXECUTE_STAGE=False",
    }
compression_summary


Grad-CAM 특징은 임베딩에 이어 붙이거나 PCA/PQ 입력으로 사용하지 않습니다.
두 결과는 다음 단계에서 표 수준으로만 결합합니다.
